# 05 Gradient Boosting Models

Trains and evaluates Gradient Boosting models for the three pollutant targets.

**Run order:** this notebook is part of the AirAware workflow and uses the shared repository folders: `data/`, `data/processed/`, and `outputs/`.

In [ ]:
# ============================================================
# 0. Setup
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import GradientBoostingRegressor, RandomForestClassifier
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    classification_report,
    confusion_matrix,
)

# ============================================================
# Project paths
# ============================================================
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

DATA_DIR.mkdir(exist_ok=True)
PROCESSED_DATA_DIR.mkdir(exist_ok=True)
OUTPUTS_DIR.mkdir(exist_ok=True)

CLEAN_FEATURED_PATH = PROCESSED_DATA_DIR / "air_quality_cleaned_featured.csv"
OUT_DIR = OUTPUTS_DIR / "gradient_boosting"
OUT_DIR.mkdir(exist_ok=True)

print("Input dataset:", CLEAN_FEATURED_PATH)
print("Output directory:", OUT_DIR)


In [ ]:
# ============================================================
# 1. Load prepared dataset
# ============================================================

df = pd.read_csv(CLEAN_FEATURED_PATH)
print(df.shape)
df.head()


In [ ]:
# Define target variables
target_cols = ["pm25_rate", "pm10_rate", "no2_rate"]

# Check whether target variables exist
for col in target_cols:
    print(col, col in df.columns)


In [ ]:
#Define features
# Targets, transformed targets and exposure variables should NOT be used as input features

exclude_cols = [
    # target variables
    "pm25_rate", "pm10_rate", "no2_rate",

    # log-transformed target variables
    "log1p_pm25_rate", "log1p_pm10_rate", "log1p_no2_rate",

    # exposure variables
    "pm25_exposure", "pm10_exposure", "no2_exposure",
    "log1p_pm25_exposure", "log1p_pm10_exposure", "log1p_no2_exposure",

    # event labels generated from target variables
    "pm25_rate_event", "pm10_rate_event", "no2_rate_event",
    "pm25_rate_high_event", "pm10_rate_high_event", "no2_rate_high_event"
]

# Keep only numeric columns and remove excluded columns
feature_cols = [
    col for col in df.select_dtypes(include=["int64", "float64", "bool"]).columns
    if col not in exclude_cols
]

print("Number of features:", len(feature_cols))
print(feature_cols)

# full model

In [ ]:
#Function for Gradient Boosting model

def train_gradient_boosting(target):
    print("=" * 60)
    print(f"Target variable: {target}")

    # Keep rows where target is not missing
    data = df[feature_cols + [target]].dropna()

    X = data[feature_cols]
    y = data[target]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42
    )

    #model = GradientBoostingRegressor(
        #n_estimators=200,
        #learning_rate=0.05,
        #max_depth=3,
        #random_state=42
    #)

    model = GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=3,
        subsample=1.0,
        min_samples_leaf=1,
        random_state=42
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    print("MAE:", round(mae, 4))
    print("RMSE:", round(rmse, 4))
    print("R²:", round(r2, 4))

    return model, X_train, X_test, y_train, y_test, y_pred

In [ ]:
#Train models separately

gb_pm25, X_train_pm25, X_test_pm25, y_train_pm25, y_test_pm25, pred_pm25 = train_gradient_boosting("pm25_rate")

gb_pm10, X_train_pm10, X_test_pm10, y_train_pm10, y_test_pm10, pred_pm10 = train_gradient_boosting("pm10_rate")

gb_no2, X_train_no2, X_test_no2, y_train_no2, y_test_no2, pred_no2 = train_gradient_boosting("no2_rate")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def get_feature_importance(model, feature_cols, top_n=15):

    importance = pd.DataFrame({
        "feature": feature_cols,
        "importance": model.feature_importances_
    }).sort_values(by="importance", ascending=False).head(top_n)

    return importance


# Get importance tables
importance_pm25 = get_feature_importance(gb_pm25, feature_cols)
importance_pm10 = get_feature_importance(gb_pm10, feature_cols)
importance_no2 = get_feature_importance(gb_no2, feature_cols)


# Create 1x3 figure
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# PM2.5
axes[0].barh(
    importance_pm25["feature"],
    importance_pm25["importance"]
)
axes[0].invert_yaxis()
axes[0].set_title("PM2.5 Model")
axes[0].set_xlabel("Feature Importance")
axes[0].set_ylabel("Feature")

# PM10
axes[1].barh(
    importance_pm10["feature"],
    importance_pm10["importance"]
)
axes[1].invert_yaxis()
axes[1].set_title("PM10 Model")
axes[1].set_xlabel("Feature Importance")
axes[1].set_ylabel("Feature")

# NO2
axes[2].barh(
    importance_no2["feature"],
    importance_no2["importance"]
)
axes[2].invert_yaxis()
axes[2].set_title("NO₂ Model")
axes[2].set_xlabel("Feature Importance")
axes[2].set_ylabel("Feature")

#fig.suptitle(
    #"Feature Importance of Gradient Boosting Models",
    #fontsize=16
#)

plt.tight_layout()

plt.show()

In [ ]:
#actual vs predicted values
import matplotlib.pyplot as plt

def plot_actual_vs_predicted(y_test, y_pred, target):
    plt.figure(figsize=(6, 6))
    plt.scatter(y_test, y_pred, alpha=0.3)
    plt.xlabel("Actual Values")
    plt.ylabel("Predicted Values")
    plt.title(f"Actual vs Predicted: {target}")

    min_val = min(y_test.min(), y_pred.min())
    max_val = max(y_test.max(), y_pred.max())
    plt.plot([min_val, max_val], [min_val, max_val])

    plt.show()

plot_actual_vs_predicted(y_test_pm25, pred_pm25, "pm25_rate")
plot_actual_vs_predicted(y_test_pm10, pred_pm10, "pm10_rate")
plot_actual_vs_predicted(y_test_no2, pred_no2, "no2_rate")

# reduce some variables

In [ ]:
features_pm25_selected = [
    "cooking_method_grilling",
    "cooking_method_frying",
    "appliance_type_electric",
    "gas_cooking_or_gas_hob",
    "gas_cooking_without_ventilation",
    "appliance_type_gas",
    "cooking_duration_min",
    "cooking_intensity",
    "extractor_fan_on",
    "ventilation_count",
    "cooking_method_oven",
    "outdoor_no2",
    "outdoor_pm25",
    "cooking_without_ventilation",
    "any_ventilation"
]

features_pm10_selected = [
    "cooking_method_grilling",
    "cooking_method_frying",
    "appliance_type_electric",
    "gas_cooking_or_gas_hob",
    "cooking_intensity",
    "appliance_type_gas",
    "cooking_duration_min",
    "gas_cooking_without_ventilation",
    "ventilation_count",
    "extractor_fan_on",
    "cooking_without_ventilation",
    "cooking_method_oven",
    "outdoor_no2",
    "appliance_type_none",
    "cooking_event"
]

features_no2_selected = [
    "gas_cooking_or_gas_hob",
    "appliance_type_gas",
    "cooking_duration_min",
    "cooking_method_frying",
    "cooking_method_oven",
    "extractor_fan_on",
    "cooking_intensity",
    "cooking_method_grilling",
    "cooking_without_ventilation",
    "ventilation_count",
    "gas_cooking_without_ventilation",
    "activity_type_cooking",
    "appliance_type_none",
    "cooking_method_none",
    "cooking_event"
]

# Check whether all selected features exist in the dataset
for name, features in {
    "PM2.5": features_pm25_selected,
    "PM10": features_pm10_selected,
    "NO2": features_no2_selected
}.items():
    missing_features = [col for col in features if col not in df.columns]
    print(name, "missing features:", missing_features)

In [ ]:
#Train Gradient Boosting with selected features
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

def train_gb_selected_features(target, selected_features):
    print("=" * 60)
    print(f"Target variable: {target}")
    print(f"Number of selected features: {len(selected_features)}")

    # Keep only selected features that exist in the dataframe
    selected_features = [col for col in selected_features if col in df.columns]

    data = df[selected_features + [target]].dropna()

    X = data[selected_features]
    y = data[target]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42
    )

    model = GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    print("MAE :", round(mae, 4))
    print("RMSE:", round(rmse, 4))
    print("R²  :", round(r2, 4))

    return model, X_train, X_test, y_train, y_test, y_pred

In [ ]:
#Retrain three target-specific models

gb_pm25_top, X_train_pm25_top, X_test_pm25_top, y_train_pm25_top, y_test_pm25_top, pred_pm25_top = train_gb_selected_features(
    "pm25_rate",
    features_pm25_selected
)

gb_pm10_top, X_train_pm10_top, X_test_pm10_top, y_train_pm10_top, y_test_pm10_top, pred_pm10_top = train_gb_selected_features(
    "pm10_rate",
    features_pm10_selected
)

gb_no2_top, X_train_no2_top, X_test_no2_top, y_train_no2_top, y_test_no2_top, pred_no2_top = train_gb_selected_features(
    "no2_rate",
    features_no2_selected
)

# Actual vs Predicted Values and Residual Analysis

In [ ]:
def plot_actual_vs_predicted_order(y_test, y_pred, target_name, n_points=300):

    import matplotlib.pyplot as plt
    import numpy as np

    y_test = np.array(y_test)
    y_pred = np.array(y_pred)

    n = min(n_points, len(y_test))

    plt.figure(figsize=(14, 5))

    # Actual values
    plt.plot(
        range(n),
        y_test[:n],
        label="Actual",
        linewidth=1.5
    )

    # Predicted values (thinner orange line)
    plt.plot(
        range(n),
        y_pred[:n],
        label="Predicted",
        linewidth=1.5
    )

    plt.xlabel("Test Sample Index")
    plt.ylabel(target_name)
    plt.title(f"Actual vs Predicted Values ({target_name})")

    plt.legend()

    plt.show()

In [ ]:
#Plot for PM2.5

plot_actual_vs_predicted_order(
    y_test_pm25,
    pred_pm25,
    "pm25_rate"
)

In [ ]:
#Plot for PM10

plot_actual_vs_predicted_order(
    y_test_pm10,
    pred_pm10,
    "pm10_rate"
)

In [ ]:
#Plot for NO2

plot_actual_vs_predicted_order(
    y_test_no2,
    pred_no2,
    "no2_rate"
)

Residual Analysis

In [ ]:
#Residual Analysis Function

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def residual_analysis(y_test, y_pred, target_name):
    y_test = np.array(y_test)
    y_pred = np.array(y_pred)

    residuals = y_test - y_pred

    residual_df = pd.DataFrame({
        "Actual": y_test,
        "Predicted": y_pred,
        "Residual": residuals
    })

    print("=" * 60)
    print(f"Residual Analysis for {target_name}")
    print(residual_df["Residual"].describe())

    # 1. Residual distribution
    plt.figure(figsize=(7, 4))
    plt.hist(residuals, bins=40)
    plt.xlabel("Residuals")
    plt.ylabel("Frequency")
    plt.title(f"Residual Distribution: {target_name}")
    plt.show()

    # 2. Predicted values vs residuals
    plt.figure(figsize=(7, 4))
    plt.scatter(y_pred, residuals, alpha=0.3)
    plt.axhline(0, linestyle="--")
    plt.xlabel("Predicted Values")
    plt.ylabel("Residuals")
    plt.title(f"Predicted Values vs Residuals: {target_name}")
    plt.show()

    # 3. Actual values vs residuals
    plt.figure(figsize=(7, 4))
    plt.scatter(y_test, residuals, alpha=0.3)
    plt.axhline(0, linestyle="--")
    plt.xlabel("Actual Values")
    plt.ylabel("Residuals")
    plt.title(f"Actual Values vs Residuals: {target_name}")
    plt.show()

    return residual_df

In [ ]:
residual_pm25 = residual_analysis(
    y_test_pm25,
    pred_pm25,
    "pm25_rate"
)

In [ ]:
residual_pm10 = residual_analysis(
    y_test_pm10,
    pred_pm10,
    "pm10_rate"
)

In [ ]:
residual_no2 = residual_analysis(
    y_test_no2,
    pred_no2,
    "no2_rate"
)

# log-transformed target

In [ ]:
#Create log-transformed target variables

import numpy as np

df["log1p_pm25_rate"] = np.log1p(df["pm25_rate"])
df["log1p_pm10_rate"] = np.log1p(df["pm10_rate"])
df["log1p_no2_rate"] = np.log1p(df["no2_rate"])

print(df[[
    "pm25_rate", "log1p_pm25_rate",
    "pm10_rate", "log1p_pm10_rate",
    "no2_rate", "log1p_no2_rate"
]].head())

In [ ]:
#Function for Gradient Boosting using log targets

from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV

def train_log_gradient_boosting(target_log, target_original):

    print("=" * 60)
    print(f"Log-transformed target: {target_log}")

    # Remove missing rows
    data = df[feature_cols + [target_log, target_original]].dropna()

    X = data[feature_cols]

    # Model trains on log target
    y_log = data[target_log]

    # Original target for evaluation
    y_original = data[target_original]

    # Train-test split
    X_train, X_test, y_train_log, y_test_log, y_train_original, y_test_original = train_test_split(
        X,
        y_log,
        y_original,
        test_size=0.2,
        random_state=42
    )

    # Base model
    gb = GradientBoostingRegressor(random_state=42)

    # Smaller parameter search space
    param_grid = {
        "n_estimators": [100, 200],
        "learning_rate": [0.03, 0.05, 0.1],
        "max_depth": [2, 3],
        "min_samples_leaf": [1, 3],
        "subsample": [0.8, 1.0]
    }

    # Randomized search
    search = RandomizedSearchCV(
        estimator=gb,
        param_distributions=param_grid,
        n_iter=8,
        scoring="neg_root_mean_squared_error",
        cv=3,
        random_state=42,
        n_jobs=-1
    )

    # Train on log target
    search.fit(X_train, y_train_log)

    # Best tuned model
    model = search.best_estimator_

    print("Best Parameters:")
    print(search.best_params_)

    # Predict LOG values
    y_pred_log = model.predict(X_test)

    # Convert back to ORIGINAL scale
    y_pred_original = np.expm1(y_pred_log)

    # Evaluation on ORIGINAL scale
    mae = mean_absolute_error(y_test_original, y_pred_original)
    rmse = np.sqrt(mean_squared_error(y_test_original, y_pred_original))
    r2 = r2_score(y_test_original, y_pred_original)

    print("MAE :", round(mae, 4))
    print("RMSE:", round(rmse, 4))
    print("R²  :", round(r2, 4))

    return (
        model,
        X_train,
        X_test,
        y_train_original,
        y_test_original,
        y_pred_original
    )

In [ ]:
#PM2.5 log model

gb_log_pm25, X_train_pm25_log, X_test_pm25_log, y_train_pm25_log, y_test_pm25_log, pred_pm25_log = train_log_gradient_boosting(
    target_log="log1p_pm25_rate",
    target_original="pm25_rate"
)

In [ ]:
#PM10 log model

gb_log_pm10, X_train_pm10_log, X_test_pm10_log, y_train_pm10_log, y_test_pm10_log, pred_pm10_log = train_log_gradient_boosting(
    target_log="log1p_pm10_rate",
    target_original="pm10_rate"
)

In [ ]:
#Train NO2 log model

gb_log_no2, X_train_no2_log, X_test_no2_log, y_train_no2_log, y_test_no2_log, pred_no2_log = train_log_gradient_boosting(
    target_log="log1p_no2_rate",
    target_original="no2_rate"
)

In [ ]:
#Plot actual vs predicted (original scale)

import matplotlib.pyplot as plt

def plot_actual_vs_predicted(y_test, y_pred, target_name, n_points=300):

    y_test = np.array(y_test)
    y_pred = np.array(y_pred)

    n = min(n_points, len(y_test))

    plt.figure(figsize=(14, 5))

    plt.plot(
        range(n),
        y_test[:n],
        label="Actual",
        linewidth=1.8
    )

    plt.plot(
        range(n),
        y_pred[:n],
        label="Predicted",
        linewidth=1.3
    )

    plt.xlabel("Test Sample Index")
    plt.ylabel(target_name)
    plt.title(f"Actual vs Predicted ({target_name})")

    plt.legend()

    plt.show()

plot_actual_vs_predicted(
    y_test_pm25_log,
    pred_pm25_log,
    "pm25_rate"
)

plot_actual_vs_predicted(
    y_test_pm10_log,
    pred_pm10_log,
    "pm10_rate"
)

plot_actual_vs_predicted(
    y_test_no2_log,
    pred_no2_log,
    "no2_rate"
)

# Two-Stage Model

In [ ]:
#Stage 1: Classify high pollution event
#Stage 2: Use separate regressors for normal / high cases

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor, RandomForestClassifier
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    classification_report,
    confusion_matrix
)

In [ ]:
# Function: Two-stage model

def train_two_stage_model(target, high_quantile=0.90):

    print("=" * 70)
    print(f"Two-stage model for: {target}")

    # Keep complete rows
    data = df[feature_cols + [target]].dropna().copy()

    # Define high pollution threshold
    threshold = data[target].quantile(high_quantile)
    data["high_event"] = (data[target] >= threshold).astype(int)

    print(f"High pollution threshold ({high_quantile} quantile): {round(threshold, 4)}")
    print(data["high_event"].value_counts())

    X = data[feature_cols]
    y = data[target]
    y_class = data["high_event"]

    # Same train-test split for classifier and regressors
    X_train, X_test, y_train, y_test, y_class_train, y_class_test = train_test_split(
        X,
        y,
        y_class,
        test_size=0.2,
        random_state=42,
        stratify=y_class
    )

    # Stage 1

    classifier = RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        random_state=42,
        class_weight="balanced"
    )

    classifier.fit(X_train, y_class_train)

    y_class_pred = classifier.predict(X_test)

    print("\nStage 1 Classification Performance:")
    print(classification_report(y_class_test, y_class_pred))
    print("Confusion Matrix:")
    print(confusion_matrix(y_class_test, y_class_pred))
    # Stage 2: Separate regressors
    # Split training data into normal and high subsets
    normal_train_idx = y_class_train == 0
    high_train_idx = y_class_train == 1

    X_train_normal = X_train[normal_train_idx]
    y_train_normal = y_train[normal_train_idx]

    X_train_high = X_train[high_train_idx]
    y_train_high = y_train[high_train_idx]

    print("\nNormal training samples:", X_train_normal.shape[0])
    print("High-event training samples:", X_train_high.shape[0])

    normal_model = GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    )

    high_model = GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    )

    normal_model.fit(X_train_normal, y_train_normal)
    high_model.fit(X_train_high, y_train_high)

    # Final prediction using classifier decision

    final_pred = np.zeros(len(X_test))

    normal_test_idx = y_class_pred == 0
    high_test_idx = y_class_pred == 1

    final_pred[normal_test_idx] = normal_model.predict(X_test[normal_test_idx])
    final_pred[high_test_idx] = high_model.predict(X_test[high_test_idx])

    # Evaluation

    mae = mean_absolute_error(y_test, final_pred)
    rmse = np.sqrt(mean_squared_error(y_test, final_pred))
    r2 = r2_score(y_test, final_pred)

    print("\nFinal Two-Stage Regression Performance:")
    print("MAE :", round(mae, 4))
    print("RMSE:", round(rmse, 4))
    print("R²  :", round(r2, 4))

    return {
        "target": target,
        "threshold": threshold,
        "classifier": classifier,
        "normal_model": normal_model,
        "high_model": high_model,
        "X_test": X_test,
        "y_test": y_test,
        "y_class_test": y_class_test,
        "y_class_pred": y_class_pred,
        "final_pred": final_pred
    }

In [ ]:
#Train two-stage models for three target variables

two_stage_pm25 = train_two_stage_model(
    target="pm25_rate",
    high_quantile=0.90
)

two_stage_pm10 = train_two_stage_model(
    target="pm10_rate",
    high_quantile=0.90
)

two_stage_no2 = train_two_stage_model(
    target="no2_rate",
    high_quantile=0.90
)

In [ ]:
two_stage_no2 = train_two_stage_model(
    target="no2_rate",
    high_quantile=0.90
)